## **Powering Sustainable Aviation: Strategic Pathways for SAF Infrastructure and Policy Incentives​**
### **IATA - IE School of Science and Technology's Sustainability Datathon**
**Group Cheerleaders:** Theo Henry, Pablo Infante, Tito Vilaplana, Bassem el Halawani, Nicolás Alejandro Higuera Wilches
___



## **1.** Introduction
This notebook walks through our full modelling approach for the IE–IATA Sustainability Datathon. The goal is to project how aviation fuel demand, SAF uptake, and CO₂ emissions could evolve for the EU27 between now and 2050 under two contrasting policy pathways: a market-driven scenario (S0) where SAF adoption follows voluntary momentum, and a policy-driven scenario (S1) aligned with the ReFuelEU mandates. We explain why these projections matter for airlines, policymakers, and investors—because they frame the scale of residual emissions, SAF demand, compliance costs, and overall decarbonisation speed. We also introduce the core datasets (Eurostat, EUROCONTROL parameters, and cost inputs) and define the scenario labels used throughout the notebook so readers can follow the logic from inputs to conclusions.

---
## **2.** Data Extraction and Transformation
This section ingests all raw inputs, Eurostat activity and energy figures, SAF pathways, ETS and fuel price assumptions—and prepares them for modelling. We standardise units (fuel in Mt, CO₂ in Mt, costs in EUR), harmonise country and year labels, and compute derived variables such as fossil vs. SAF fuel, fuel-per-flight, and emissions based on the standard ICAO/IPCC factor of 3.16 tCO₂ per tonne of fuel. This step also handles any data cleaning needed: filling occasional missing values, aligning time ranges, and ensuring that both scenarios share consistent baselines. By the end of this section, all inputs feed seamlessly into the projection engine.

### **2.1.** Notebook setup

Imports, plotting defaults, and pandas display settings so the rest of the notebook runs cleanly.

In [22]:
# Core imports for data ingestion, analysis, and plotting
import gzip
import io
from datetime import date
from typing import Dict, Iterable, List
import inspect

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 50)


### **2.2.** Assumptions, constraints and configuration

#### **a.** Baseline & Scope
We use EU27 historical data from Eurostat (2000–2023) for:
	•	Total aviation fuel consumption (ktoe → Mt),
	•	Flights (commercial IFR movements),
	•	SAF share,
	•	Supporting activity variables.

Our 2023 fuel baseline is ~71.5 Mt of aviation fuel consumed in the EU27.
This baseline aligns with Eurostat’s territorial accounting but not with the lower ReFuelEU policy scope (≈39 Mt); therefore we use the Eurostat scope consistently throughout.

All emissions use the ICAO/IPCC standard emission factor:

EF = 3.16 tCO₂ per tonne of fuel.


#### **b.** Traffic Growth (EUROCONTROL Base Scenario)

To remain consistent with EUROCONTROL’s long-term outlook, we use their Base Scenario annual flight growth rates:
•	2024–2030: +2.5% per year
•	2030–2040: +1.4% per year
•	2040–2050: +1.1% per year

These values reflect expected post-COVID recovery, stabilization, and long-term maturity of the European aviation market.


#### **c.** Efficiency Improvements

Fuel demand does not grow at the same pace as flights because more efficient aircraft enter the fleet and ATM/operational procedures improve gradually. In our model, efficiency is implemented as yearly multiplicative reductions.

**Fleet (technology) efficiency**
We assume EUROCONTROL-aligned baseline fleet improvement of:

~1% per year
→ ≈ 20% cumulative fleet efficiency gain to 2050

This choice places us at the middle of EUROCONTROL’s 9–32% range for cumulative fleet efficiency, rather than at the high end (which would require ~1.5%/yr and produce a 33–34% gain).

**Operational efficiency**
Operational (ATM/SESAR) efficiency improves more slowly:
•	2024–2035: 0.2% per year
•	2036–2050: 0.1% per year

This yields a ~4% cumulative operational improvement by 2050.


#### **d.** Fuel Demand Projection

We use a net fuel model where annual fuel demand evolves as:

$F_t = F_{t-1}$
$\times (1 + g_\text{flights})$
$\times (1 - g_\text{fleet})$
$\times (1 - g_\text{ops})$

This means:
•	Traffic pushes fuel demand up,
•	Efficiency improvements push it down,
•	The net effect is a relatively flat long-run fuel trajectory for the EU27, which matches EUROCONTROL’s expectation that efficiency roughly offsets traffic growth in Europe over the long term.

Fuel never includes SAF effects — SAF only affects emissions, not fuel mass.


#### **e.** SAF Scenarios (Two Policy Pathways)

We model two scenarios for SAF blending share:

**S0 — Market / Business-as-Usual (no mandates)**
A slow voluntary uptake aligned with EASA, ATAG, and IATA baseline expectations:
	•	2025: 0.6%
	•	2030: 2%
	•	2035: 6%
	•	2050: 35%

**S1 — ReFuelEU / Policy Scenario (EU mandates)**
We apply the actual ReFuelEU trajectory:
	•	2025: 2%
	•	2030: 6%
	•	2035: 20%
	•	2040: 42%
	•	2050: 70%

Between milestone years, we use linear interpolation to create a smooth annual SAF curve.


#### **f.** Emissions Calculation

For each year and scenario:

**CO₂ exhaust (no SAF, no reductions)**

$$\text{CO₂}_{\text{exhaust}} = F_t \times 3.16$$


**CO₂ net (with SAF lifecycle reduction):** SAF reduces lifecycle CO₂ by ≈75%, the midpoint of recognized values (EASA/IATA range 65–80%):

$$\text{CO₂}_{\text{net}} = \text{CO₂}_{\text{exhaust}} \times (1 - s_t \times 0.75)$$

**CO₂ avoided**

$$\text{CO₂}_{\text{avoided}} = \text{CO₂}_{\text{exhaust}} - \text{CO₂}_{\text{net}}.$$

In [23]:
# Shared geography lists, time windows, and conversion/policy constants

EU27_TOP5_STRING = "EU27_2020,DE,ES,FR,IT,NL"
ALL_GEOS_STRING = "BE,BG,CZ,DK,DE,EE,IE,EL,ES,FR,HR,IT,CY,LV,LT,LU,HU,MT,NL,AT,PL,PT,RO,SI,SK,FI,SE"
TARGET_GEOS = ["EU27", "DE", "ES", "FR", "IT", "NL"]
GEO_MAP = {
    "Austria": "AT", "Belgium": "BE", "Bulgaria": "BG", "Croatia": "HR", "Cyprus": "CY",
    "Czechia": "CZ", "Denmark": "DK", "Estonia": "EE", "Finland": "FI", "France": "FR",
    "Germany": "DE", "Greece": "EL", "Hungary": "HU", "Ireland": "IE", "Italy": "IT",
    "Latvia": "LV", "Lithuania": "LT", "Luxembourg": "LU", "Malta": "MT", "Netherlands": "NL",
    "Poland": "PL", "Portugal": "PT", "Romania": "RO", "Slovakia": "SK", "Slovenia": "SI",
    "Spain": "ES", "Sweden": "SE", "European Union - 27 countries (from 2020)": "EU27", "EU27_2020": "EU27",
}

HIST_START, HIST_END = 2010, 2023
FORECAST_START, FORECAST_END = 2024, 2050
FORECAST_RANGE = np.arange(FORECAST_START, FORECAST_END + 1)

KTOE_TO_TONNES = 11630  # ktoe -> tonnes
EF_CO2 = 3.16  # t CO2 / t fuel
SAF_REDUCTION = 0.75  # emissions reduction from SAF share
EU27_POLICY_SCALE = 38.8 / 71.685785  # align Eurostat fuel to policy baseline
MT_TO_TONNES = 1e6

S0_MILESTONES = {2025: 0.006, 2030: 0.020, 2035: 0.060, 2050: 0.350}
S1_MILESTONES = {2025: 0.020, 2030: 0.060, 2035: 0.200, 2040: 0.420, 2050: 0.700}


### **2.3.** Data collection structuring

Functions to pull fuel balances, flights, passengers, freight, and fleet age/mix from Eurostat and normalize units.

In [24]:
# Helper to download CSV data (handles optional gzip compression)

def fetch_csv(url: str) -> pd.DataFrame:
    """Download CSV (optionally gzip compressed) and return as dataframe."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    raw = resp.content
    try:
        data_bytes = gzip.decompress(raw)
    except gzip.BadGzipFile:
        data_bytes = raw
    return pd.read_csv(io.BytesIO(data_bytes))

In [25]:
# Pull and clean Eurostat datasets for fuel, flights, passengers, freight, and fleet mix

def pull_df_fuel() -> pd.DataFrame:
    url = (
        "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/nrg_bal_c/1.0/*.*.*.*.*"
        f"?c[freq]=A&c[nrg_bal]=FC_TRA_DAVI_E&c[siec]=O4661XR5230B,R5230P,R5230B&c[unit]=KTOE&c[geo]={EU27_TOP5_STRING}"
        f"&c[TIME_PERIOD]=ge:{HIST_START}+le:{HIST_END}"
        "&compress=true&format=csvdata&formatVersion=2.0&lang=en&labels=name"
    )
    df = fetch_csv(url)
    df = df[["geo", "TIME_PERIOD", "siec", "OBS_VALUE"]].rename(columns={"TIME_PERIOD": "year", "OBS_VALUE": "value"})
    df = df.pivot_table(index=["geo", "year"], columns="siec", values="value", aggfunc="first").reset_index()
    df["geo"] = df["geo"].replace({"EU27_2020": "EU27"})
    rename_map = {"O4661XR5230B": "ktoe_fossil_fuel", "R5230P": "ktoe_blended_fuel", "R5230B": "ktoe_bio_fuel"}
    df = df.rename(columns=rename_map)
    for col in ["ktoe_fossil_fuel", "ktoe_blended_fuel", "ktoe_bio_fuel"]:
        series = df[col] if col in df else pd.Series(0, index=df.index)
        df[col] = pd.to_numeric(series, errors="coerce").fillna(0.0)
        series = df[col] if col in df else pd.Series(0, index=df.index)
        df[col] = pd.to_numeric(series, errors="coerce").fillna(0.0)
    df["ktoe_total_fuel"] = df["ktoe_fossil_fuel"] + df["ktoe_blended_fuel"] + df["ktoe_bio_fuel"]
    df["mt_fossil_fuel"] = df["ktoe_fossil_fuel"] * KTOE_TO_TONNES / 1e6
    df["mt_blended_fuel"] = df["ktoe_blended_fuel"] * KTOE_TO_TONNES / 1e6
    df["mt_bio_fuel"] = df["ktoe_bio_fuel"] * KTOE_TO_TONNES / 1e6
    df["mt_total_fuel"] = df["ktoe_total_fuel"] * KTOE_TO_TONNES / 1e6
    df["saf_share"] = (df["ktoe_bio_fuel"] + df["ktoe_blended_fuel"]) / df["ktoe_total_fuel"].replace(0, pd.NA)
    return df


def pull_df_flights() -> pd.DataFrame:
    url = (
        "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/avia_tf_acc/1.0/*.*.*.*.*.*"
        f"?c[freq]=A&c[unit]=FLIGHT&c[tra_meas]=CAF&c[tra_cov]=TOTAL&c[aircraft]=TOTAL&c[geo]={ALL_GEOS_STRING}"
        f"&c[TIME_PERIOD]=ge:{HIST_START}+le:{HIST_END}"
        "&compress=false&format=csvdata&formatVersion=1.0&lang=en&labels=label_only"
    )
    df = fetch_csv(url)
    df = df[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={"TIME_PERIOD": "year", "OBS_VALUE": "flights"})
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["flights"] = pd.to_numeric(df["flights"], errors="coerce")
    df = df.groupby(["geo", "year"], as_index=False)["flights"].sum().dropna(subset=["year"])
    df["geo"] = df["geo"].replace(GEO_MAP)
    eu27_rows = df.groupby("year", as_index=False)["flights"].sum().assign(geo="EU27")
    eu27_rows = eu27_rows.reindex(columns=df.columns)
    df = pd.concat([df, eu27_rows], ignore_index=True)
    df = df[df["geo"].isin(TARGET_GEOS)].sort_values(["geo", "year"]).reset_index(drop=True)
    df["flights_yoy"] = df.groupby("geo")["flights"].pct_change()
    return df


def pull_df_passengers() -> pd.DataFrame:
    url = (
        "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/avia_paoc/1.0/*.*.*.*.*.*"
        f"?c[freq]=A&c[unit]=PAS&c[tra_meas]=PAS_CRD&c[tra_cov]=TOTAL&c[schedule]=TOT&c[geo]={EU27_TOP5_STRING}"
        f"&c[TIME_PERIOD]=ge:{HIST_START}+le:{HIST_END}"
        "&compress=false&format=csvdata&formatVersion=1.0&lang=en&labels=label_only"
    )
    df = fetch_csv(url)
    df = df[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={"TIME_PERIOD": "year", "OBS_VALUE": "passengers"})
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["passengers"] = pd.to_numeric(df["passengers"], errors="coerce")
    df = df.groupby(["geo", "year"], as_index=False)["passengers"].sum().dropna(subset=["year"])
    df["geo"] = df["geo"].replace(GEO_MAP)
    df = df.sort_values(["geo", "year"]).reset_index(drop=True)
    df["passengers_yoy"] = df.groupby("geo")["passengers"].pct_change()
    return df


def pull_df_freight() -> pd.DataFrame:
    url = (
        "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/avia_gooc/1.0/*.*.*.*.*.*"
        f"?c[freq]=A&c[unit]=T&c[tra_meas]=FRM_BRD&c[schedule]=TOT&c[tra_cov]=TOTAL&c[geo]={EU27_TOP5_STRING}"
        f"&c[TIME_PERIOD]=ge:{HIST_START}+le:{HIST_END}"
        "&compress=false&format=csvdata&formatVersion=1.0&lang=en&labels=label_only"
    )
    df = fetch_csv(url)
    df = df[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={"TIME_PERIOD": "year", "OBS_VALUE": "freight_tonnes"})
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["freight_tonnes"] = pd.to_numeric(df["freight_tonnes"], errors="coerce")
    df = df.groupby(["geo", "year"], as_index=False)["freight_tonnes"].sum().dropna(subset=["year"])
    df["geo"] = df["geo"].replace(GEO_MAP)
    df = df.sort_values(["geo", "year"]).reset_index(drop=True)
    df["freight_tonnes_yoy"] = df.groupby("geo")["freight_tonnes"].pct_change()
    return df


def pull_df_fleet() -> pd.DataFrame:
    url = (
        "https://ec.europa.eu/eurostat/api/dissemination/sdmx/3.0/data/dataflow/ESTAT/avia_eq_arc_age/1.0/*.*.*"
        f"?c[freq]=A&c[age]=Y_LT5,Y5-9,Y10-14,Y15-19,Y_GE20&c[geo]={ALL_GEOS_STRING}"
        f"&c[TIME_PERIOD]=ge:{HIST_START}+le:{HIST_END}"
        "&compress=false&format=csvdata&formatVersion=1.0&lang=en&labels=label_only"
    )
    df = fetch_csv(url)
    df = df[["geo", "age", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={"TIME_PERIOD": "year", "OBS_VALUE": "aircraft"})
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    df["aircraft"] = pd.to_numeric(df["aircraft"], errors="coerce")
    df = df.pivot_table(index=["geo", "year"], columns="age", values="aircraft", aggfunc="first").reset_index()
    rename_map = {"Less than 5 years": "air_lt5", "From 5 to 9 years": "air_5_9", "From 10 to 14 years": "air_10_14", "From 15 to 19 years": "air_15_19", "20 years or over": "air_20plus"}
    df = df.rename(columns=rename_map)
    for col in rename_map.values():
        df[col] = pd.to_numeric(df.get(col, 0.0), errors="coerce").fillna(0.0)
    df["total_fleet"] = df["air_lt5"] + df["air_5_9"] + df["air_10_14"] + df["air_15_19"] + df["air_20plus"]
    df["modern_fleet"] = df["air_lt5"] + df["air_5_9"]
    df = df[["geo", "year", "total_fleet", "modern_fleet"]]
    df["geo"] = df["geo"].replace(GEO_MAP)
    eu27_rows = df.groupby("year", as_index=False)[["total_fleet", "modern_fleet"]].sum().assign(geo="EU27")
    eu27_rows = eu27_rows.reindex(columns=df.columns)
    df = pd.concat([df, eu27_rows], ignore_index=True)
    df = df[df["geo"].isin(TARGET_GEOS)].sort_values(["geo", "year"]).reset_index(drop=True)
    df["modern_fleet_share"] = df["modern_fleet"] / df["total_fleet"].replace(0, pd.NA)
    return df


### **2.4.** Data collection and grouping
Execute the fetchers to materialize raw inputs (fuel, flights, passengers, freight, fleet) for later merges.

Combine fuel, traffic, and fleet datasets into one tidy table keyed by geography and year, with derived fuel/SAF fields.

In [26]:
# Execute data pulls so downstream sections can merge and project

df_fuel = pull_df_fuel()
df_flights = pull_df_flights()
df_passengers = pull_df_passengers()
df_freight = pull_df_freight()
df_fleet = pull_df_fleet()

print("Fuel rows", len(df_fuel))
print("Flights rows", len(df_flights))
print("Passengers rows", len(df_passengers))
print("Freight rows", len(df_freight))
print("Fleet rows", len(df_fleet))

df_fuel.head()


Fuel rows 84
Flights rows 84
Passengers rows 84
Freight rows 84
Fleet rows 84


siec,geo,year,ktoe_fossil_fuel,ktoe_bio_fuel,ktoe_blended_fuel,ktoe_total_fuel,mt_fossil_fuel,mt_blended_fuel,mt_bio_fuel,mt_total_fuel,saf_share
0,DE,2010,772.829,0.0,0,772.829,8.988001,0.0,0.0,8.988001,0.0
1,DE,2011,709.449,0.0,0,709.449,8.250892,0.0,0.0,8.250892,0.0
2,DE,2012,710.471,0.0,0,710.471,8.262778,0.0,0.0,8.262778,0.0
3,DE,2013,666.514,0.0,0,666.514,7.751558,0.0,0.0,7.751558,0.0
4,DE,2014,715.582,0.0,0,715.582,8.322219,0.0,0.0,8.322219,0.0


In [27]:
# Merge fuel/traffic/fleet sources into a single core dataframe keyed by geo and year

def merge_core_sources(df_fuel, df_flights, df_passengers, df_freight, df_fleet) -> pd.DataFrame:
    df = df_fuel.copy()
    df = pd.merge(
        df[["geo", "year", "mt_fossil_fuel", "mt_blended_fuel", "mt_bio_fuel", "mt_total_fuel", "saf_share"]],
        df_flights[["geo", "year", "flights", "flights_yoy"]],
        on=["geo", "year"],
        how="inner",
    )
    df = pd.merge(df, df_passengers[["geo", "year", "passengers", "passengers_yoy"]], on=["geo", "year"], how="inner")
    df = pd.merge(df, df_freight[["geo", "year", "freight_tonnes", "freight_tonnes_yoy"]], on=["geo", "year"], how="inner")
    df = pd.merge(df, df_fleet[["geo", "year", "total_fleet", "modern_fleet", "modern_fleet_share"]], on=["geo", "year"], how="inner")
    df = df.sort_values(["geo", "year"]).reset_index(drop=True)
    return df[df["geo"] == "EU27"].reset_index(drop=True)

df_eu27 = merge_core_sources(df_fuel, df_flights, df_passengers, df_freight, df_fleet)
df_eu27.head()


,geo,year,mt_fossil_fuel,mt_blended_fuel,mt_bio_fuel,mt_total_fuel,saf_share,flights,flights_yoy,passengers,passengers_yoy,freight_tonnes,freight_tonnes_yoy,total_fleet,modern_fleet,modern_fleet_share
0,EU27,2010,64.504772,0.0,0.0,64.504772,0.0,9052561,NaN,687720194,NaN,11400983.2,NaN,5500,2972,0.540364
1,EU27,2011,68.762282,0.0,0.0,68.762282,0.0,9418645,0.040440,730656231,0.062432,11888796.8,0.042787,5531,2929,0.529561
2,EU27,2012,63.853585,0.0,0.0,63.853585,0.0,9077695,-0.036199,734860381,0.005754,11517678.0,-0.031216,5401,2882,0.533605
3,EU27,2013,59.646723,0.0,0.0,59.646723,0.0,8847328,-0.025377,746100398,0.015295,11558357.0,0.003532,5239,2808,0.535980
4,EU27,2014,59.943718,0.0,0.0,59.943718,0.0,8927557,0.009068,781202599,0.047048,12511005.7,0.082421,5254,2761,0.525504


### **2.5.** Projection design: Forecasting method and parameters selection

Helper functions to project flights, fuel demand, and SAF share trajectories under different scenarios.

In [28]:
# Scenario toggle for sensitivities (0=pessimistic, 1=baseline, 2=optimistic)

# control scenario: pessimistic=0, baseline=1, optimistic=2
scenario_toggle = 1


# 1. flight growth rates based on scenario
def flights_growth(year: int) -> float:
    if scenario_toggle == 0: # pessimistic EUROCONTROL notes that in the Low scenario, traffic grows only ~+20% between 2023 and 2050, which corresponds to ~0.7%/yr.
        return 0.007

    elif scenario_toggle == 1: # baseline
        if year <= 2030:
            return 0.025
        if year <= 2040:
            return 0.014
        return 0.010
    elif scenario_toggle == 2: # optimistic
        if year <= 2030:
            return 0.040
        if year <= 2040:
            return 0.018
        return 0.014


# 2. fleet efficiency rates based on scenario
def fleet_eff_improvement(year: int) -> float:
    #return 0.015 if year <= 2040 else 0.01
    if scenario_toggle == 0: # pessimistic
        return 0.005
    elif scenario_toggle == 1: # baseline
        return 0.009
    elif scenario_toggle == 2: # optimistic
        return 0.01


# 3. operational efficiency rates based on scenario
def ops_eff_improvement(year: int) -> float:
    if scenario_toggle == 0: # pessimistic
        return 0.001
    elif scenario_toggle == 1: # baseline
        return 0.002 if year <= 2035 else 0.001
    elif scenario_toggle == 2: # optimistic
        return 0.002

# flight projection
def project_flights(flights_last: float, years: Iterable[int]) -> pd.Series:
    out, current = [], flights_last
    for y in years:
        current *= 1 + flights_growth(y)
        out.append(current)
    return pd.Series(out, index=years)

# fuel projection with no improvement to fleet or operations
def project_fuel_gross(mt_total_fuel_last: float, years: Iterable[int]):
    F = mt_total_fuel_last
    out = []
    for y in years:
        F *= (1 + flights_growth(y)) # ONLY traffic growth
        out.append(F)
    return pd.Series(out, index=years)

# fuel projection with improvements to fleet and operations
def project_fuel_net(mt_total_fuel_last: float, years: Iterable[int]) -> pd.Series:
    out, current = [], mt_total_fuel_last
    for y in years:
        current *= (1 + flights_growth(y)) * (1 - fleet_eff_improvement(y)) * (1 - ops_eff_improvement(y))
        out.append(current)
    return pd.Series(out, index=years)

# saf implementation path
def saf_path(last_share: float, years: Iterable[int], milestones: Dict[int, float]) -> pd.Series:
    years = list(years)
    key_years = [years[0] - 1] + sorted(milestones)
    key_values = [last_share] + [milestones[y] for y in sorted(milestones)]
    return pd.Series(np.interp(years, key_years, key_values), index=years)


---
## **3.** Core metrics projection 2024-2050 (Objective 1)
Here we build the heart of the model: long-run projections for flights, fuel demand, SAF share, CO₂ exhaust, CO₂ net, and avoided emissions. Flights grow according to EUROCONTROL’s Base Scenario rates (higher in the 2020s, slower after 2030), while fuel evolves using a net-fuel model that blends traffic growth with annual fleet and operational efficiency improvements. 

SAF uptake follows two explicit adoption curves: the conservative S0_market trajectory (0.6% → 35%) and the ambitious S1_policy mandate pathway (2% → 70%). CO₂ emissions are then computed mechanically from fuel using EF = 3.16 and applying a 75% SAF lifecycle reduction. This gives us a transparent, reproducible set of emissions paths that show how much of the decarbonisation comes from efficiency versus SAF, and how far policy accelerates progress relative to business-as-usual.

In [29]:
# Build scenario-specific trajectories for traffic, efficiency, and SAF share

def make_scenario(df_hist: pd.DataFrame, scenario: str, milestones: Dict[int, float]) -> pd.DataFrame:
    blocks: List[pd.DataFrame] = []
    for geo, gdf in df_hist.groupby("geo"):
        gdf = gdf.sort_values("year")
        if HIST_END not in set(gdf["year"]):
            continue
        flights_last = int(gdf.loc[gdf["year"] == HIST_END, "flights"].iloc[0])
        fuel_last = float(gdf.loc[gdf["year"] == HIST_END, "mt_total_fuel"].iloc[0])
        saf_last = float(gdf.loc[gdf["year"] == HIST_END, "saf_share"].iloc[0])

        flight_proj = project_flights(flights_last, FORECAST_RANGE)
        fuel_proj_gross = project_fuel_gross(fuel_last, FORECAST_RANGE)
        fuel_proj_net = project_fuel_net(fuel_last, FORECAST_RANGE)
        saf_proj = saf_path(saf_last, FORECAST_RANGE, milestones)

        df_proj = pd.DataFrame({
            "geo": geo,
            "year": FORECAST_RANGE,
            "flights": flight_proj.values,
            "mt_total_fuel": fuel_proj_net.values,
            "saf_share": saf_proj.values,
            "scenario": scenario,
        })
        df_proj["mt_co2_gross"] = fuel_proj_gross* EF_CO2 #without efficiency improvements
        df_proj["mt_co2_baseline"] = df_proj["mt_total_fuel"] * EF_CO2 #with efficiency improvements
        df_proj["mt_co2_emissions"] = df_proj["mt_co2_baseline"] * (1 - df_proj["saf_share"] * SAF_REDUCTION) # with saf reduction
        df_proj["mt_co2_avoided"] = df_proj["mt_co2_baseline"] - df_proj["mt_co2_emissions"] # avoided emissions from improvements and SAF
        blocks.append(df_proj)
    return pd.concat(blocks, ignore_index=True) if blocks else pd.DataFrame()


def build_core_metrics(df_eu27: pd.DataFrame) -> pd.DataFrame:
    df_hist = df_eu27.loc[df_eu27["year"] <= HIST_END, ["geo", "year", "mt_total_fuel", "saf_share", "flights"]].copy()
    df_hist["mt_total_fuel"] = df_hist["mt_total_fuel"] * EU27_POLICY_SCALE

    df_hist_e = df_hist.copy()
    df_hist_e["scenario"] = "Historical"
    df_hist_e["mt_co2_gross"] = df_hist_e["mt_total_fuel"] * EF_CO2
    df_hist_e["mt_co2_baseline"] = df_hist_e["mt_total_fuel"] * EF_CO2
    df_hist_e["mt_co2_emissions"] = df_hist_e["mt_co2_baseline"] * (1 - df_hist_e["saf_share"].fillna(0) * SAF_REDUCTION)
    df_hist_e["mt_co2_avoided"] = df_hist_e["mt_co2_baseline"] - df_hist_e["mt_co2_emissions"]

    s0 = make_scenario(df_hist, "S0_market", S0_MILESTONES)
    s1 = make_scenario(df_hist, "S1_policy", S1_MILESTONES)

    core = pd.concat([df_hist_e, s0, s1], ignore_index=True)
    core = core.sort_values(["geo", "year", "scenario"]).reset_index(drop=True)
    core = core[["geo", "year", "scenario", "flights", "mt_total_fuel", "saf_share","mt_co2_baseline" , "mt_co2_emissions", "mt_co2_avoided"]]
    return core

core = build_core_metrics(df_eu27)
core.to_csv("objective1_projections.csv", index=False)


### **3.1.** 'Business as Usual' scenario projection 2024-2050

In [30]:
print("Market projection:")
display(core[core['scenario']=='S0_market'].head(len(core[core['scenario']=='S0_market'])))
df_obj1_s0 = core[core['scenario']=='S0_market']
df_obj1_s0.to_csv('objective1_market_projection.csv', index=False)

Market projection:


,geo,year,scenario,flights,mt_total_fuel,saf_share,mt_co2_baseline,mt_co2_emissions,mt_co2_avoided
14,EU27,2024,S0_market,9.805093e+06,39.333246,0.004047,124.293057,123.915759,0.377297
16,EU27,2025,S0_market,1.005022e+07,39.873820,0.006000,126.001272,125.434266,0.567006
18,EU27,2026,S0_market,1.030148e+07,40.421824,0.008800,127.732964,126.889927,0.843038
20,EU27,2027,S0_market,1.055901e+07,40.977359,0.011600,129.488456,128.361906,1.126550
22,EU27,2028,S0_market,1.082299e+07,41.540530,0.014400,131.268074,129.850379,1.417695
24,EU27,2029,S0_market,1.109356e+07,42.111440,0.017200,133.072150,131.355519,1.716631
26,EU27,2030,S0_market,1.137090e+07,42.690196,0.020000,134.901021,132.877505,2.023515
28,EU27,2031,S0_market,1.153009e+07,42.812472,0.028000,135.287411,132.446375,2.841036
30,EU27,2032,S0_market,1.169152e+07,42.935098,0.036000,135.674908,132.011686,3.663223
32,EU27,2033,S0_market,1.185520e+07,43.058075,0.044000,136.063515,131.573419,4.490096


### **3.2.** Policy-driven scenario projection 2024-2050

In [31]:
print("Policy projection:")
display(core[core['scenario']=='S1_policy'].head(len(core[core['scenario']=='S1_policy'])))
df_obj1_s1 = core[core['scenario']=='S1_policy']
df_obj1_s1.to_csv('objective1_policy_projection.csv', index=False)

Policy projection:


,geo,year,scenario,flights,mt_total_fuel,saf_share,mt_co2_baseline,mt_co2_emissions,mt_co2_avoided
15,EU27,2024,S1_policy,9.805093e+06,39.333246,0.011047,124.293057,123.263221,1.029836
17,EU27,2025,S1_policy,1.005022e+07,39.873820,0.020000,126.001272,124.111253,1.890019
19,EU27,2026,S1_policy,1.030148e+07,40.421824,0.028000,127.732964,125.050572,2.682392
21,EU27,2027,S1_policy,1.055901e+07,40.977359,0.036000,129.488456,125.992268,3.496188
23,EU27,2028,S1_policy,1.082299e+07,41.540530,0.044000,131.268074,126.936228,4.331846
25,EU27,2029,S1_policy,1.109356e+07,42.111440,0.052000,133.072150,127.882336,5.189814
27,EU27,2030,S1_policy,1.137090e+07,42.690196,0.060000,134.901021,128.830475,6.070546
29,EU27,2031,S1_policy,1.153009e+07,42.812472,0.088000,135.287411,126.358442,8.928969
31,EU27,2032,S1_policy,1.169152e+07,42.935098,0.116000,135.674908,123.871191,11.803717
33,EU27,2033,S1_policy,1.185520e+07,43.058075,0.144000,136.063515,121.368656,14.694860


___
## **4.** Economic and policy implications (Objective 2)
With the physical trajectories in place, this section translates them into euro terms. We estimate fossil and SAF fuel expenditure, apply ETS carbon cost assumptions, and compute the marginal abatement cost (MAC) of SAF by dividing extra fuel spending by avoided CO₂. The analysis highlights how policy affects financial outcomes for example, S1_policy reduces emissions far more, but requires higher SAF volumes and therefore higher near-term costs. 

This section also incorporates any included price inputs (fossil prices, SAF premiums, carbon prices) exactly as provided in the challenge guidelines or assumptions. If CORSIA costs are zero in the dataset (as in this version), they are treated explicitly as zero to avoid ambiguity. The result is a clear picture of the cost-benefit trade-off under each scenario.

In [32]:
# Slice EU27 data for Objective 2, map scenarios to numeric labels, and keep 2025-2050

df = core.query("geo == 'EU27' and 2025 <= year <= 2050 and scenario in ['S0_market', 'S1_policy']").copy()
df["Scenario"] = df["scenario"].map({"S0_market": 0, "S1_policy": 1})
df = df.rename(columns={
    "year": "Year",
    "mt_total_fuel": "Total_Fuel",
    "saf_share": "SAF_Share",
    "mt_co2_emissions": "CO2_Emissions",
    "mt_co2_avoided": "Avoided_CO2",
})
if df["SAF_Share"].max() > 1.01:
    df["SAF_Share"] = df["SAF_Share"] / 100.0
assert df["Total_Fuel"].min() > 0
assert set(df["Scenario"].unique()) <= {0, 1}
display(df.head(12))
display(df.tail(12))


,geo,Year,scenario,flights,Total_Fuel,SAF_Share,mt_co2_baseline,CO2_Emissions,Avoided_CO2,Scenario
16,EU27,2025,S0_market,1.005022e+07,39.873820,0.0060,126.001272,125.434266,0.567006,0
17,EU27,2025,S1_policy,1.005022e+07,39.873820,0.0200,126.001272,124.111253,1.890019,1
18,EU27,2026,S0_market,1.030148e+07,40.421824,0.0088,127.732964,126.889927,0.843038,0
19,EU27,2026,S1_policy,1.030148e+07,40.421824,0.0280,127.732964,125.050572,2.682392,1
20,EU27,2027,S0_market,1.055901e+07,40.977359,0.0116,129.488456,128.361906,1.126550,0
21,EU27,2027,S1_policy,1.055901e+07,40.977359,0.0360,129.488456,125.992268,3.496188,1
22,EU27,2028,S0_market,1.082299e+07,41.540530,0.0144,131.268074,129.850379,1.417695,0
23,EU27,2028,S1_policy,1.082299e+07,41.540530,0.0440,131.268074,126.936228,4.331846,1
24,EU27,2029,S0_market,1.109356e+07,42.111440,0.0172,133.072150,131.355519,1.716631,0
25,EU27,2029,S1_policy,1.109356e+07,42.111440,0.0520,133.072150,127.882336,5.189814,1


,geo,Year,scenario,flights,Total_Fuel,SAF_Share,mt_co2_baseline,CO2_Emissions,Avoided_CO2,Scenario
56,EU27,2045,S0_market,1.373350e+07,44.129294,0.253333,139.448568,112.953340,26.495228,0
57,EU27,2045,S1_policy,1.373350e+07,44.129294,0.560000,139.448568,80.880170,58.568399,1
58,EU27,2046,S0_market,1.387084e+07,44.125282,0.272667,139.435891,110.921251,28.514640,0
59,EU27,2046,S1_policy,1.387084e+07,44.125282,0.588000,139.435891,77.944663,61.491228,1
60,EU27,2047,S0_market,1.400955e+07,44.121271,0.292000,139.423215,108.889531,30.533684,0
61,EU27,2047,S1_policy,1.400955e+07,44.121271,0.616000,139.423215,75.009690,64.413525,1
62,EU27,2048,S0_market,1.414964e+07,44.117260,0.311333,139.410540,106.858179,32.552361,0
63,EU27,2048,S1_policy,1.414964e+07,44.117260,0.644000,139.410540,72.075249,67.335291,1
64,EU27,2049,S0_market,1.429114e+07,44.113249,0.330667,139.397866,104.827195,34.570671,0
65,EU27,2049,S1_policy,1.429114e+07,44.113249,0.672000,139.397866,69.141342,70.256525,1


### **4.1.** Economic assumptions

Central/low/high inputs for fuel prices, ETS carbon price, SAF premium, and traffic splits. ETS = EU Emissions Trading System; it sets the carbon price used later in the cost model.

In [33]:
# Economic assumption table for fuel prices, ETS, SAF premium, demand splits, and references

def build_assumptions() -> pd.DataFrame:
    data = [
        {"name": "Jet_A1_price_2024", "segment": "system", "unit": "EUR/t fuel", "central": 734, "low": 734 * 0.7, "high": 734 * 1.3, "source": "EASA 2024 Aviation Fuels Reference Prices", "year": 2024, "link": "https://www.easa.europa.eu/en/downloads/141675/en", "note": "Conventional aviation fuel (CAF) reference price, used as Jet A-1 proxy."},
        {"name": "SAF_price_2024", "segment": "system", "unit": "EUR/t fuel", "central": 2085, "low": 2085 * 0.7, "high": 2085 * 1.3, "source": "EASA 2024 Aviation Fuels Reference Prices", "year": 2024, "link": "https://www.easa.europa.eu/en/downloads/141675/en", "note": "Weighted average of SAF categories (excl. synthetic) in 2024."},
        {"name": "SAF_premium", "segment": "system", "unit": "EUR/t fuel", "central": 2085 - 734, "low": (2085 - 734) * 0.7, "high": (2085 - 734) * 1.3, "source": "Derived from EASA 2024 CAF vs SAF reference prices", "year": 2024, "link": "https://www.easa.europa.eu/en/downloads/141675/en", "note": "Average SAF premium vs fossil jet; used as constant real premium."},
        {"name": "ETS_price_2030", "segment": "system", "unit": "EUR/tCO2", "central": 72, "low": 72 * 0.7, "high": 72 * 1.3, "source": "Enerdata / EC ETS forecast + Carbon Market Watch synthesis", "year": 2030, "link": "https://www.enerdata.net/publications/executive-briefing/carbon-price-forecast-under-eu-ets.html", "note": "Order-of-magnitude ETS allowance price around 2030; used to anchor path."},
        {"name": "SAF_LCA_reduction", "segment": "system", "unit": "% vs fossil", "central": 0.80, "low": 0.70, "high": 0.90, "source": "HEFA / SAF life-cycle studies; IATA net-zero materials", "year": 2024, "link": "https://doi.org/10.1016/j.apenergy.2024.122946", "note": "Average life-cycle CO2 reduction of SAF vs fossil jet; aligns with 76-97% range."},
        {"name": "Passenger_share_CO2", "segment": "split", "unit": "share of CO2/fuel", "central": 0.85, "low": 0.80, "high": 0.90, "source": "ICCT / commercial aviation CO2 split", "year": 2019, "link": "https://theicct.org/publication/co2-commercial-aviation-2013-2018-2019/", "note": "Approximate share of commercial aviation CO2 from passenger traffic."},
        {"name": "EU_passengers_2023", "segment": "passenger", "unit": "million passengers", "central": 973, "low": 900, "high": 1050, "source": "Eurostat air passenger statistics", "year": 2023, "link": "https://ec.europa.eu/eurostat/statistics-explained/index.php/Air_passenger_transport_statistics", "note": "Used to derive passengers for cost pass-through, scaled by fuel demand."},
        {"name": "EU_air_freight_2023", "segment": "freight", "unit": "million tonnes", "central": 13.1, "low": 12, "high": 15, "source": "Eurostat air freight & mail transport", "year": 2023, "link": "https://ec.europa.eu/eurostat/statistics-explained/index.php/Air_freight_transport_statistics", "note": "Used to derive freight tonnes for cost pass-through."},
    ]
    return pd.DataFrame(data)

assumptions_df = build_assumptions()
#display(assumptions_df)


### **4.2.** Price paths

Simple real-terms paths for jet fuel, ETS carbon price, and SAF premium with optional shocks (e.g., ±30%). ETS is the regulated carbon price; shocks let us test upside/downside cases.

In [34]:
# Simple price path helpers for jet fuel, ETS carbon price, and SAF premium shocks

def jet_price_path(year: int, shock: float = 0.0, jet_base_2024: float = 734.0) -> float:
    return jet_base_2024 * (1.0 + shock)

def ets_price_path(year: int, shock: float = 0.0) -> float:
    start, end = 65.0, 100.0
    frac = (year - 2025) / (2050 - 2025)
    price = start + frac * (end - start)
    return price * (1.0 + shock)

def saf_premium_path(year: int, shock: float = 0.0, saf_premium_central: float = 1351.0) -> float:
    return max(0.0, saf_premium_central * (1.0 + shock))


### **4.3.** Passenger/freight segmentation

Split total fuel/CO₂ into passenger vs freight segments based on assumed shares, scaling to match system fuel totals.

In [35]:
# Split system fuel/CO₂ into passenger vs freight segments and align volumes

def build_segments(df: pd.DataFrame, assumptions_df: pd.DataFrame) -> pd.DataFrame:
    pax_share = assumptions_df.loc[assumptions_df["name"] == "Passenger_share_CO2", "central"].iloc[0]
    freight_share = 1.0 - pax_share
    base_pax_2023 = assumptions_df.loc[assumptions_df["name"] == "EU_passengers_2023", "central"].iloc[0] * 1e6
    base_freight_2023 = assumptions_df.loc[assumptions_df["name"] == "EU_air_freight_2023", "central"].iloc[0] * 1e6

    fuel_2025 = df.loc[df["Year"] == 2025, "Total_Fuel"].mean()
    fuel_ref = fuel_2025 if fuel_2025 > 0 else 1.0

    pax_2025 = base_pax_2023 * (df.loc[(df["Year"] == 2025) & (df["Scenario"] == 0), "Total_Fuel"].iloc[0] / fuel_ref)
    freight_2025 = base_freight_2023 * (df.loc[(df["Year"] == 2025) & (df["Scenario"] == 0), "Total_Fuel"].iloc[0] / fuel_ref)

    segments = []
    for seg, share in [("Passenger", pax_share), ("Freight", freight_share)]:
        tmp = df.copy()
        tmp["Segment"] = seg
        tmp["Fuel_Mt_segment"] = tmp["Total_Fuel"] * share
        tmp["CO2_Mt_segment"] = tmp["CO2_Emissions"] * share
        segments.append(tmp)

    df_seg = pd.concat(segments, ignore_index=True)
    df_seg["Passengers"] = np.where(
        df_seg["Segment"] == "Passenger",
        pax_2025 * (df_seg["Total_Fuel"] / df.loc[(df["Year"] == 2025) & (df["Scenario"] == 0), "Total_Fuel"].iloc[0]),
        0.0,
    )
    df_seg["Freight_tonnes"] = np.where(
        df_seg["Segment"] == "Freight",
        freight_2025 * (df_seg["Total_Fuel"] / df.loc[(df["Year"] == 2025) & (df["Scenario"] == 0), "Total_Fuel"].iloc[0]),
        0.0,
    )
    return df_seg

df_seg = build_segments(df, assumptions_df)


### **4.4.** Cost model (central case)

Apply price paths to fuel/SAF split to compute fuel expenditure, ETS cost, total cost, and CO₂ by year and scenario. ETS adds a carbon fee on fossil fuel; SAF premium captures the price delta vs Jet A-1.

In [36]:
# Cost model: compute fuel split, SAF premium, ETS cost, and total system cost for each scenario

def compute_costs(df_in: pd.DataFrame, saf_premium_shock: float = 0.0, ets_price_shock: float = 0.0, jet_price_shock: float = 0.0, saf_cfd_credit_eur_per_tonne: float = 0.0, ets_rebate_share: float = 0.0, jet_base_2024: float = 734.0, saf_premium_central: float = 1351.0) -> pd.DataFrame:
    df = df_in.copy()
    df["Jet_fuel_Mt"] = df["Total_Fuel"] * (1.0 - df["SAF_Share"])
    df["SAF_Mt"] = df["Total_Fuel"] * df["SAF_Share"]
    df["Jet_fuel_t"] = df["Jet_fuel_Mt"] * MT_TO_TONNES
    df["SAF_t"] = df["SAF_Mt"] * MT_TO_TONNES

    jet_prices, saf_prices, ets_prices = [], [], []
    fuel_exp_list, ets_cost_list, total_cost_list = [], [], []

    for _, row in df.iterrows():
        year = int(row["Year"])
        jet_price = jet_price_path(year, shock=jet_price_shock, jet_base_2024=jet_base_2024)
        saf_premium = saf_premium_path(year, shock=saf_premium_shock, saf_premium_central=saf_premium_central)
        saf_premium_eff = max(0.0, saf_premium - saf_cfd_credit_eur_per_tonne)
        saf_price = jet_price + saf_premium_eff
        ets_price = ets_price_path(year, shock=ets_price_shock)

        fuel_exp = row["Jet_fuel_t"] * jet_price + row["SAF_t"] * saf_price
        ets_cost = row["CO2_Emissions"] * MT_TO_TONNES * ets_price
        ets_cost_adj = ets_cost * (1.0 - ets_rebate_share)

        jet_prices.append(jet_price)
        saf_prices.append(saf_price)
        ets_prices.append(ets_price)
        fuel_exp_list.append(fuel_exp)
        ets_cost_list.append(ets_cost_adj)
        total_cost_list.append(fuel_exp + ets_cost_adj)

    df["Jet_price_eur_per_t"] = jet_prices
    df["SAF_price_eur_per_t"] = saf_prices
    df["ETS_price_eur_per_tCO2"] = ets_prices
    df["Fuel_Exp_EUR"] = fuel_exp_list
    df["ETS_Cost_EUR"] = ets_cost_list
    df["Total_Cost_EUR"] = total_cost_list
    return df

jet_base = assumptions_df.loc[assumptions_df["name"] == "Jet_A1_price_2024", "central"].iloc[0]
saf_premium_central = assumptions_df.loc[assumptions_df["name"] == "SAF_premium", "central"].iloc[0]

df_central = compute_costs(df_seg, jet_base_2024=jet_base, saf_premium_central=saf_premium_central)
print("Cost model output sample 2025-2030 (central case)")
display(df_central.head(12))
print("Cost model output sample 2045-2050 (central case)")
display(df_central.tail(12))


Cost model output sample 2025-2030 (central case)


,geo,Year,scenario,flights,Total_Fuel,SAF_Share,mt_co2_baseline,CO2_Emissions,Avoided_CO2,Scenario,Segment,Fuel_Mt_segment,CO2_Mt_segment,Passengers,Freight_tonnes,Jet_fuel_Mt,SAF_Mt,Jet_fuel_t,SAF_t,Jet_price_eur_per_t,SAF_price_eur_per_t,ETS_price_eur_per_tCO2,Fuel_Exp_EUR,ETS_Cost_EUR,Total_Cost_EUR
0,EU27,2025,S0_market,1.005022e+07,39.873820,0.0060,126.001272,125.434266,0.567006,0,Passenger,33.892747,106.619126,9.730000e+08,0.0,39.634577,0.239243,3.963458e+07,2.392429e+05,734.0,2085.0,65.0,2.959060e+10,8.153227e+09,3.774383e+10
1,EU27,2025,S1_policy,1.005022e+07,39.873820,0.0200,126.001272,124.111253,1.890019,1,Passenger,33.892747,105.494565,9.730000e+08,0.0,39.076344,0.797476,3.907634e+07,7.974764e+05,734.0,2085.0,65.0,3.034477e+10,8.067231e+09,3.841201e+10
2,EU27,2026,S0_market,1.030148e+07,40.421824,0.0088,127.732964,126.889927,0.843038,0,Passenger,34.358551,107.856438,9.863724e+08,0.0,40.066112,0.355712,4.006611e+07,3.557121e+05,734.0,2085.0,66.4,3.015019e+10,8.425491e+09,3.857568e+10
3,EU27,2026,S1_policy,1.030148e+07,40.421824,0.0280,127.732964,125.050572,2.682392,1,Passenger,34.358551,106.292986,9.863724e+08,0.0,39.290013,1.131811,3.929001e+07,1.131811e+06,734.0,2085.0,66.4,3.119870e+10,8.303358e+09,3.950205e+10
4,EU27,2027,S0_market,1.055901e+07,40.977359,0.0116,129.488456,128.361906,1.126550,0,Passenger,34.830756,109.107620,9.999285e+08,0.0,40.502022,0.475337,4.050202e+07,4.753374e+05,734.0,2085.0,67.8,3.071956e+10,8.702937e+09,3.942250e+10
5,EU27,2027,S1_policy,1.055901e+07,40.977359,0.0360,129.488456,125.992268,3.496188,1,Passenger,34.830756,107.093427,9.999285e+08,0.0,39.502175,1.475185,3.950217e+07,1.475185e+06,734.0,2085.0,67.8,3.207036e+10,8.542276e+09,4.061263e+10
6,EU27,2028,S0_market,1.082299e+07,41.540530,0.0144,131.268074,129.850379,1.417695,0,Passenger,35.309450,110.372822,1.013671e+09,0.0,40.942346,0.598184,4.094235e+07,5.981836e+05,734.0,2085.0,69.2,3.129889e+10,8.985646e+09,4.028454e+10
7,EU27,2028,S1_policy,1.082299e+07,41.540530,0.0440,131.268074,126.936228,4.331846,1,Passenger,35.309450,107.895793,1.013671e+09,0.0,39.712746,1.827783,3.971275e+07,1.827783e+06,734.0,2085.0,69.2,3.296008e+10,8.783987e+09,4.174407e+10
8,EU27,2029,S0_market,1.109356e+07,42.111440,0.0172,133.072150,131.355519,1.716631,0,Passenger,35.794724,111.652192,1.027602e+09,0.0,41.387123,0.724317,4.138712e+07,7.243168e+05,734.0,2085.0,70.6,3.188835e+10,9.273700e+09,4.116205e+10
9,EU27,2029,S1_policy,1.109356e+07,42.111440,0.0520,133.072150,127.882336,5.189814,1,Passenger,35.794724,108.699986,1.027602e+09,0.0,39.921645,2.189795,3.992165e+07,2.189795e+06,734.0,2085.0,70.6,3.386821e+10,9.028493e+09,4.289670e+10


Cost model output sample 2045-2050 (central case)


,geo,Year,scenario,flights,Total_Fuel,SAF_Share,mt_co2_baseline,CO2_Emissions,Avoided_CO2,Scenario,Segment,Fuel_Mt_segment,CO2_Mt_segment,Passengers,Freight_tonnes,Jet_fuel_Mt,SAF_Mt,Jet_fuel_t,SAF_t,Jet_price_eur_per_t,SAF_price_eur_per_t,ETS_price_eur_per_tCO2,Fuel_Exp_EUR,ETS_Cost_EUR,Total_Cost_EUR
92,EU27,2045,S0_market,1.373350e+07,44.129294,0.253333,139.448568,112.953340,26.495228,0,Freight,6.619394,16.943001,0.0,1.449808e+07,32.949873,11.179421,3.294987e+07,1.117942e+07,734.0,2085.0,93.0,4.749430e+10,1.050466e+10,5.799896e+10
93,EU27,2045,S1_policy,1.373350e+07,44.129294,0.560000,139.448568,80.880170,58.568399,1,Freight,6.619394,12.132025,0.0,1.449808e+07,19.416889,24.712405,1.941689e+07,2.471240e+07,734.0,2085.0,93.0,6.577736e+10,7.521856e+09,7.329922e+10
94,EU27,2046,S0_market,1.387084e+07,44.125282,0.272667,139.435891,110.921251,28.514640,0,Freight,6.618792,16.638188,0.0,1.449676e+07,32.093788,12.031494,3.209379e+07,1.203149e+07,734.0,2085.0,94.4,4.864250e+10,1.047097e+10,5.911347e+10
95,EU27,2046,S1_policy,1.387084e+07,44.125282,0.588000,139.435891,77.944663,61.491228,1,Freight,6.618792,11.691699,0.0,1.449676e+07,18.179616,25.945666,1.817962e+07,2.594567e+07,734.0,2085.0,94.4,6.744055e+10,7.357976e+09,7.479853e+10
96,EU27,2047,S0_market,1.400955e+07,44.121271,0.292000,139.423215,108.889531,30.533684,0,Freight,6.618191,16.333430,0.0,1.449544e+07,31.237860,12.883411,3.123786e+07,1.288341e+07,734.0,2085.0,95.8,4.979050e+10,1.043162e+10,6.022212e+10
97,EU27,2047,S1_policy,1.400955e+07,44.121271,0.616000,139.423215,75.009690,64.413525,1,Freight,6.618191,11.251453,0.0,1.449544e+07,16.942568,27.178703,1.694257e+07,2.717870e+07,734.0,2085.0,95.8,6.910344e+10,7.185928e+09,7.628937e+10
98,EU27,2048,S0_market,1.414964e+07,44.117260,0.311333,139.410540,106.858179,32.552361,0,Freight,6.617589,16.028727,0.0,1.449412e+07,30.382086,13.735173,3.038209e+07,1.373517e+07,734.0,2085.0,97.2,5.093829e+10,1.038661e+10,6.132490e+10
99,EU27,2048,S1_policy,1.414964e+07,44.117260,0.644000,139.410540,72.075249,67.335291,1,Freight,6.617589,10.811287,0.0,1.449412e+07,15.705744,28.411515,1.570574e+07,2.841152e+07,734.0,2085.0,97.2,7.076603e+10,7.005714e+09,7.777174e+10
100,EU27,2049,S0_market,1.429114e+07,44.113249,0.330667,139.397866,104.827195,34.570671,0,Freight,6.616987,15.724079,0.0,1.449281e+07,29.526468,14.586781,2.952647e+07,1.458678e+07,734.0,2085.0,98.6,5.208587e+10,1.033596e+10,6.242183e+10
101,EU27,2049,S1_policy,1.429114e+07,44.113249,0.672000,139.397866,69.141342,70.256525,1,Freight,6.616987,10.371201,0.0,1.449281e+07,14.469146,29.644103,1.446915e+07,2.964410e+07,734.0,2085.0,98.6,7.242831e+10,6.817336e+09,7.924564e+10


### **4.5.** MAC and pass-through calculations
Calculate system-level marginal abatement cost (MAC = extra cost per tonne of CO₂ avoided when moving from S0 to S1) and simple pass-through estimates per passenger and per tonne of freight.

In [37]:
# System-level marginal abatement cost (MAC) and per-passenger/per-freight pass-through

def system_mac(df_costs: pd.DataFrame) -> pd.DataFrame:
    system = df_costs.groupby(["Year", "Scenario"], as_index=False).agg({
        "Total_Cost_EUR": "sum",
        "CO2_Emissions": "mean",
        "Avoided_CO2": "mean",
    })
    pivot = system.pivot(index="Year", columns="Scenario", values=["Total_Cost_EUR", "CO2_Emissions", "Avoided_CO2"])
    pivot.columns = [f"{metric}_S{sc}" for metric, sc in pivot.columns]
    pivot = pivot.reset_index()
    pivot["Delta_Cost_EUR"] = pivot["Total_Cost_EUR_S1"] - pivot["Total_Cost_EUR_S0"]
    pivot["Delta_Avoided_CO2_Mt"] = pivot["Avoided_CO2_S1"] - pivot["Avoided_CO2_S0"]
    pivot["MAC_EUR_per_tCO2"] = pivot["Delta_Cost_EUR"] / (pivot["Delta_Avoided_CO2_Mt"] * MT_TO_TONNES)
    return pivot

mac_central = system_mac(df_central)
display(mac_central[["Year", "MAC_EUR_per_tCO2"]].head())

df_passenger = df_central[df_central["Segment"] == "Passenger"].copy()
df_freight_seg = df_central[df_central["Segment"] == "Freight"].copy()

pax_agg = df_passenger.groupby(["Year", "Scenario"], as_index=False).agg({"Total_Cost_EUR": "sum", "Passengers": "sum"})
freight_agg = df_freight_seg.groupby(["Year", "Scenario"], as_index=False).agg({"Total_Cost_EUR": "sum", "Freight_tonnes": "sum"})

pax_pivot = pax_agg.pivot(index="Year", columns="Scenario", values=["Total_Cost_EUR", "Passengers"])
pax_pivot.columns = [f"{metric}_S{sc}" for metric, sc in pax_pivot.columns]
pax_pivot = pax_pivot.reset_index()
pax_pivot["Delta_Cost_EUR"] = pax_pivot["Total_Cost_EUR_S1"] - pax_pivot["Total_Cost_EUR_S0"]
pax_pivot["Delta_cost_per_pax_EUR"] = pax_pivot["Delta_Cost_EUR"] / pax_pivot["Passengers_S1"]

fre_pivot = freight_agg.pivot(index="Year", columns="Scenario", values=["Total_Cost_EUR", "Freight_tonnes"])
fre_pivot.columns = [f"{metric}_S{sc}" for metric, sc in fre_pivot.columns]
fre_pivot = fre_pivot.reset_index()
fre_pivot["Delta_Cost_EUR"] = fre_pivot["Total_Cost_EUR_S1"] - fre_pivot["Total_Cost_EUR_S0"]
fre_pivot["Delta_cost_per_tonne_EUR"] = fre_pivot["Delta_Cost_EUR"] / fre_pivot["Freight_tonnes_S1"]

print("Passenger cost pass-through (head):")
display(pax_pivot[["Year", "Delta_cost_per_pax_EUR"]].head())
print("Freight cost pass-through (head):")
display(fre_pivot[["Year", "Delta_cost_per_tonne_EUR"]].head())


,Year,MAC_EUR_per_tCO2
0,2025,1010.084388
1,2026,1007.284388
2,2027,1004.484388
3,2028,1001.684388
4,2029,998.884388


Passenger cost pass-through (head):


,Year,Delta_cost_per_pax_EUR
0,2025,0.686719
1,2026,0.939175
2,2027,1.190218
3,2028,1.439846
4,2029,1.688060


Freight cost pass-through (head):


,Year,Delta_cost_per_tonne_EUR
0,2025,51.005921
1,2026,69.757070
2,2027,88.403186
3,2028,106.944269
4,2029,125.380319


### **4.6.** Sensitivities (+/-30% SAF premium and ETS price)

Re-run the cost model with low/high price shocks to see how MAC and avoided CO₂ respond to SAF and ETS price swings.

In [38]:
# Sensitivity runs applying low/high shocks to SAF premium and ETS price

def run_sensitivity(df_seg: pd.DataFrame, case_label: str, saf_shock: float, ets_shock: float) -> pd.DataFrame:
    df_s = compute_costs(df_seg, saf_premium_shock=saf_shock, ets_price_shock=ets_shock, jet_base_2024=jet_base, saf_premium_central=saf_premium_central)
    sys = df_s.groupby(["Year", "Scenario"], as_index=False).agg({"Total_Cost_EUR": "sum", "Avoided_CO2": "mean"})
    piv = sys.pivot(index="Year", columns="Scenario", values=["Total_Cost_EUR", "Avoided_CO2"])
    piv.columns = [f"{metric}_S{sc}" for metric, sc in piv.columns]
    piv = piv.reset_index()
    piv["Delta_Cost_EUR"] = piv["Total_Cost_EUR_S1"] - piv["Total_Cost_EUR_S0"]
    piv["Delta_Avoided_CO2_Mt"] = piv["Avoided_CO2_S1"] - piv["Avoided_CO2_S0"]
    piv["MAC_EUR_per_tCO2"] = piv["Delta_Cost_EUR"] / (piv["Delta_Avoided_CO2_Mt"] * MT_TO_TONNES)
    piv["Case"] = case_label
    return piv[["Year", "Case", "MAC_EUR_per_tCO2"]]

sens_central = run_sensitivity(df_seg, "central", saf_shock=0.0, ets_shock=0.0)
sens_low = run_sensitivity(df_seg, "low_prices", saf_shock=-0.3, ets_shock=-0.3)
sens_high = run_sensitivity(df_seg, "high_prices", saf_shock=0.3, ets_shock=0.3)

mac_sens = pd.concat([sens_central, sens_low, sens_high], ignore_index=True)
mac_sens.head()


,Year,Case,MAC_EUR_per_tCO2
0,2025,central,1010.084388
1,2026,central,1007.284388
2,2027,central,1004.484388
3,2028,central,1001.684388
4,2029,central,998.884388


### **4.7.** Tornado inputs (2035, 2050)

Build small tables to feed tornado charts showing which drivers move MAC the most in key years.

In [39]:
# Prepare tornado-style summary rows for a target year

def tornado_data(mac_sens: pd.DataFrame, target_year: int) -> pd.DataFrame:
    base = mac_sens[(mac_sens["Year"] == target_year) & (mac_sens["Case"] == "central")]["MAC_EUR_per_tCO2"].iloc[0]
    low = mac_sens[(mac_sens["Year"] == target_year) & (mac_sens["Case"] == "low_prices")]["MAC_EUR_per_tCO2"].iloc[0]
    high = mac_sens[(mac_sens["Year"] == target_year) & (mac_sens["Case"] == "high_prices")]["MAC_EUR_per_tCO2"].iloc[0]
    return pd.DataFrame({"Parameter": ["SAF premium & ETS (-30%)", "Base", "SAF premium & ETS (+30%)"], "MAC": [low, base, high]})


tornado_2035 = tornado_data(mac_sens, 2035)
tornado_2050 = tornado_data(mac_sens, 2050)
print(tornado_2035)
print(tornado_2050)


                  Parameter          MAC
0  SAF premium & ETS (-30%)   687.459072
1                      Base   982.084388
2  SAF premium & ETS (+30%)  1276.709705
                  Parameter          MAC
0  SAF premium & ETS (-30%)   658.059072
1                      Base   940.084388
2  SAF premium & ETS (+30%)  1222.109705


---
## **5.** Data visualization

In [40]:

# Standardized Plotly visualizations (section 5)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Color palette and labels (as specified)
COLOR_MARKET = "#F85C08"
COLOR_POLICY = "#003399"
COLOR_EXHAUST = "#FFC400"
COLOR_NET = "#8EC41A"
COLOR_AVOIDED = "#1AC47A"
COLOR_FLIGHTS = "#00A1DE"
COLOR_OPS = "#FFC400"
COLOR_FLEET = "#00A1DE"

SCENARIO_NAMES = {0: "Market", 1: "Policy"}
SCENARIO_COLORS = {0: COLOR_MARKET, 1: COLOR_POLICY}
EF = 3.16
FIG_W, FIG_H = 1100, 600


def apply_common_layout(fig, title, x_title, y_title, y_title_secondary=None):
    """Apply shared sizing, fonts, background, grid, and axis styling."""
    fig.update_layout(
        title=title,
        width=FIG_W,
        height=FIG_H,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(family="Arial", size=14, color="black"),
        legend=dict(orientation="h", y=-0.2, bgcolor="rgba(0,0,0,0)", font=dict(color="black")),
        margin=dict(l=70, r=70, t=70, b=70),
    )
    fig.update_xaxes(
        title_text=x_title,
        gridcolor="lightgray",
        showline=True,
        linecolor="lightgray",
        tickmode="linear",
        dtick=5,
        zeroline=False,
    )
    if y_title_secondary:
        fig.update_yaxes(title_text=y_title, gridcolor="lightgray", showline=True, linecolor="lightgray", zeroline=False, secondary_y=False)
        fig.update_yaxes(title_text=y_title_secondary, gridcolor="lightgray", showline=True, linecolor="lightgray", zeroline=False, secondary_y=True)
    else:
        fig.update_yaxes(title_text=y_title, gridcolor="lightgray", showline=True, linecolor="lightgray", zeroline=False)
    return fig


def scenario_line(x, y, name, color, dash=None):
    """Unified line+marker style (continuous line, circle markers, width 1)."""
    return go.Scatter(
        x=x,
        y=y,
        name=name,
        mode="lines+markers",
        line=dict(color=color, width=1, dash=dash),
        marker=dict(symbol="circle", size=6, color=color),
    )


# Aggregate scenario metrics used across visuals
agg = df_central.groupby(["Year", "Scenario"], as_index=False).agg({
    "Total_Fuel": "mean",
    "SAF_Share": "mean",
    "CO2_Emissions": "mean",
    "Avoided_CO2": "mean",
    "ETS_Cost_EUR": "sum",
})
agg = agg[(agg["Year"] >= 2024) & (agg["Year"] <= 2050)]
agg["SAF_Mt"] = agg["Total_Fuel"] * agg["SAF_Share"]
agg["CO2_exhaust"] = agg["Total_Fuel"] * EF
agg["CO2_net_avoided"] = agg["CO2_exhaust"] - agg["CO2_Emissions"]
agg["Scenario_Label"] = agg["Scenario"].map(SCENARIO_NAMES)

# Baseline exhaust across scenarios (single line)
exhaust_baseline = (
    df_central.groupby("Year", as_index=False)["Total_Fuel"].mean()
    .rename(columns={"Total_Fuel": "Total_Fuel_mean"})
)
exhaust_baseline = exhaust_baseline[(exhaust_baseline["Year"] >= 2024) & (exhaust_baseline["Year"] <= 2050)]
exhaust_baseline["CO2_exhaust"] = exhaust_baseline["Total_Fuel_mean"] * EF

# Flights (policy only) to avoid double-counting; use stored value directly
flights_ts = None
if "flights" in df_central.columns:
    flights_ts = (
        df_central[df_central["Scenario"] == 1][["Year", "flights"]]
        .sort_values("Year")
        .drop_duplicates(subset=["Year"], keep="last")
    )
elif "core" in globals() and hasattr(core, "columns") and "flights" in core.columns:
    flights_ts = (
        core[core["scenario"] == "S1_policy"]
        .rename(columns={"year": "Year"})
        .sort_values("Year")
        .drop_duplicates(subset=["Year"], keep="last")
    )
if flights_ts is not None:
    flights_ts = flights_ts[(flights_ts["Year"] >= 2024) & (flights_ts["Year"] <= 2050)]
    flights_ts["Flights_int"] = flights_ts["flights"].round().astype(int)
    flights_ts["Flights_M"] = flights_ts["flights"] / 1e6

#0. Historical fuel demand (2010–2050)
hist_fuel = df_eu27[df_eu27['year']<=2050]["mt_total_fuel"]

hist_fuel = (
    core.query("2010 <= year <= 2050")[["year", "mt_total_fuel"]]
    .sort_values("year")
)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=hist_fuel["year"],
        y=hist_fuel["mt_total_fuel"],
        mode="lines+markers",
        name="Fuel demand",
        line=dict(color="#003399", width=1),
        marker=dict(symbol="circle", size=6, color="#003399"),
    )
)
apply_common_layout(fig, "Historical fuel demand (2010–2050)", "Year", "Fuel (Mt)")
fig.show()


# 1) CO2 emissions (Mt) with flights (count) on secondary axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
# Flights: single bar trace (secondary axis) using policy scenario values
if flights_ts is not None and not flights_ts.empty:
    fig.add_trace(
        go.Bar(
            x=flights_ts["Year"],
            y=flights_ts["Flights_int"],
            name="Flights",
            marker_color=COLOR_FLIGHTS,
            opacity=0.6,
        ),
        secondary_y=True,
    )
# CO2 emissions lines per scenario (primary axis)
for sc in sorted(agg["Scenario"].unique()):
    df_sc = agg[agg["Scenario"] == sc]
    fig.add_trace(
        scenario_line(
            df_sc["Year"],
            df_sc["CO2_Emissions"],
            f"CO₂: {SCENARIO_NAMES[sc]}",
            SCENARIO_COLORS[sc],
        ),
        secondary_y=False,
    )
apply_common_layout(fig, "EU27 CO₂ Emissions by Scenario vs. Flight Volume (2024-2050)", "Year", "CO₂ emissions (Mt)", "Flights (count)")
fig.show()
#---

# 2) SAF Share of total Jet Fuel (Mt)
fig = go.Figure()
for sc in sorted(agg["Scenario"].unique()):
    df_sc = agg[agg["Scenario"] == sc]
    fig.add_trace(scenario_line(df_sc["Year"], df_sc["SAF_Share"], f"SAF Share: {SCENARIO_NAMES[sc]}", SCENARIO_COLORS[sc]))
apply_common_layout(fig, "EU27 SAF Share of Total Jet Fuel (Mt) by Scenario (2024-2050)", "Year", "SAF fuel (Mt)")
fig.show()


# 3) CO2 avoided (Mt)
fig = go.Figure()
for sc in sorted(agg["Scenario"].unique()):
    df_sc = agg[agg["Scenario"] == sc]
    dash = None if sc == 0 else "dash"
    fig.add_trace(scenario_line(df_sc["Year"], df_sc["CO2_net_avoided"], f"CO₂ avoided: {SCENARIO_NAMES[sc]}", SCENARIO_COLORS[sc], dash=dash))
apply_common_layout(fig, "EU27 CO₂ Emissions Avoided by Scenario (2024-2050)", "Year", "CO₂ avoided (Mt)")
fig.show()

# 4) CO2 exhaust vs net CO2 avoided (single exhaust line)
fig = go.Figure()
fig.add_trace(scenario_line(exhaust_baseline["Year"], exhaust_baseline["CO2_exhaust"], "CO₂ exhaust", COLOR_EXHAUST))
for sc in sorted(agg["Scenario"].unique()):
    df_sc = agg[agg["Scenario"] == sc]
    dash = None if sc == 0 else "dash"
    fig.add_trace(scenario_line(df_sc["Year"], df_sc["CO2_net_avoided"], f"Net CO₂ avoided: {SCENARIO_NAMES[sc]}", SCENARIO_COLORS[sc], dash=dash))
apply_common_layout(fig, "EU27 CO₂ Exhaust vs. Net Avoided  by Scenario (2024-2050)", "Year", "CO₂ (Mt)")
fig.show()

# 5) Cumulative CO2 exhaust vs net CO2 avoided
fig = go.Figure()
#exhaust_baseline_sorted = exhaust_baseline.sort_values("Year").copy()
#exhaust_baseline_sorted["cum_exhaust"] = exhaust_baseline_sorted["CO2_exhaust"].cumsum()
#fig.add_trace(scenario_line(exhaust_baseline_sorted["Year"], exhaust_baseline_sorted["cum_exhaust"], "Cumulative CO₂ exhaust", COLOR_EXHAUST))
for sc in sorted(agg["Scenario"].unique()):
    df_sc = agg[agg["Scenario"] == sc].sort_values("Year").copy()
    df_sc["cum_avoided"] = df_sc["CO2_net_avoided"].cumsum()
    dash = None if sc == 0 else "dash"
    fig.add_trace(scenario_line(df_sc["Year"], df_sc["cum_avoided"], f"Cumulative CO₂ avoided: {SCENARIO_NAMES[sc]}", SCENARIO_COLORS[sc], dash=dash))
apply_common_layout(fig, "EU27 Cumulative CO₂ Emissions Avoided by Scenario (2024-2050)", "Year", "Cumulative CO₂ (Mt)")
fig.show()

# 6) ETS cost progression (EUR billion)
fig = go.Figure()
for sc in sorted(agg["Scenario"].unique()):
    df_sc = agg[agg["Scenario"] == sc]
    fig.add_trace(scenario_line(df_sc["Year"], df_sc["ETS_Cost_EUR"] / 1e9, f"ETS cost: {SCENARIO_NAMES[sc]}", SCENARIO_COLORS[sc]))
apply_common_layout(fig, "ETS Cost Progression by Scenario (2024-2050)", "Year", "ETS cost (EUR billion)")
fig.show()

import plotly.graph_objects as go

# Prep costs by year/scenario
costs = (
    df_central.query("2024 <= Year <= 2050")
    .groupby(["Year", "Scenario"], as_index=False)[["Fuel_Exp_EUR", "ETS_Cost_EUR"]]
    .sum()
)

SCEN_LABELS = {0: "Market", 1: "Policy"}
COLOR_FUEL = "#F85C08"
COLOR_ETS = "#FFC400"

for sc in sorted(costs["Scenario"].unique()):
    df_sc = costs[costs["Scenario"] == sc]

    fig = go.Figure()
    # Base: fuel expenditure
    fig.add_trace(
        go.Scatter(
            x=df_sc["Year"],
            y=df_sc["Fuel_Exp_EUR"] / 1e9,
            mode="lines",
            stackgroup="cost",
            name="Fuel expenditure",
            line=dict(color=COLOR_FUEL, width=1),
            hovertemplate="Year %{x}<br>Fuel €%{y:.2f}B<extra></extra>",
        )
    )
    # Top: ETS cost
    fig.add_trace(
        go.Scatter(
            x=df_sc["Year"],
            y=df_sc["ETS_Cost_EUR"] / 1e9,
            mode="lines",
            stackgroup="cost",
            name="ETS cost",
            line=dict(color=COLOR_ETS, width=1),
            hovertemplate="Year %{x}<br>ETS €%{y:.2f}B<extra></extra>",
        )
    )

    apply_common_layout(
        fig,
        f"Total cost breakdown — {SCEN_LABELS.get(sc, sc)} (Fuel + ETS)",
        "Year",
        "Cost (EUR billion)",
    )
    fig.show()


# 6.1. MAC cost progression (€/tCO₂) by scenario
# MAC cost progression (€/tCO₂) — single series from mac_central
mac_ts = (
    mac_central.groupby("Year", as_index=False)["MAC_EUR_per_tCO2"]
    .mean()
    .query("2024 <= Year <= 2050")
)

fig = go.Figure()
fig.add_trace(
    scenario_line(
        mac_ts["Year"],
        mac_ts["MAC_EUR_per_tCO2"],
        "MAC (€/tCO₂)",
        COLOR_POLICY,  # single line color; keep palette consistent
    )
)
apply_common_layout(fig, "MAC Cost Progression (2024-2050)", "Year", "MAC (€/tCO₂)")
fig.show()



# 7) Waterfall charts for 2050 (one per scenario)
for sc in sorted(agg["Scenario"].unique()):
    row = agg[(agg["Year"] == 2050) & (agg["Scenario"] == sc)]
    if row.empty:
        continue
    r = row.iloc[0]
    exhaust = r["CO2_exhaust"]
    net = r["CO2_Emissions"]
    avoided = r["CO2_net_avoided"]

    fig = go.Figure(go.Waterfall(
        x=["CO₂ exhaust", "SAF CO₂ avoided", "CO₂ net emissions"],
        measure=["absolute", "relative", "total"],
        y=[exhaust, -avoided, net],
        connector=dict(line=dict(color="lightgray")),
        increasing=dict(marker=dict(color=COLOR_EXHAUST)),
        decreasing=dict(marker=dict(color=COLOR_AVOIDED)),
        totals=dict(marker=dict(color=COLOR_NET)),
        name=SCENARIO_NAMES[sc],
    ))
    apply_common_layout(fig, f"2050 CO₂ Waterfall — {SCENARIO_NAMES[sc]}", "CO₂ components", "CO₂ (Mt)")
    fig.show()

# 8) Tornado chart for 2050 fuel demand drivers
fuel_tornado_2050 = pd.DataFrame({
    "Driver": ["Flight growth", "Operations efficiency", "Fleet efficiency"],
    "Delta_Mt": [8.0, -2.0, -6.0],
}).sort_values("Delta_Mt")
driver_colors = {
    "Flight growth": COLOR_MARKET,
    "Operations efficiency": COLOR_OPS,
    "Fleet efficiency": COLOR_FLEET,
}
fig = go.Figure(go.Bar(
    x=fuel_tornado_2050["Delta_Mt"],
    y=fuel_tornado_2050["Driver"],
    orientation="h",
    marker_color=[driver_colors[d] for d in fuel_tornado_2050["Driver"]],
    hovertemplate="%{y}: %{x:.1f} Mt<extra></extra>",
))
fig.add_vline(x=0, line_color="lightgray", line_dash="dash")
apply_common_layout(fig, "2050 Fuel Demand Drivers", "Δ fuel (Mt)", "Driver")
fig.show()


## **6.** Export outputs
Save out system-level costs, MAC, and by-segment costs/assumptions so results can be reused outside the notebook.

In [41]:
# Aggregate costs/MAC by scenario and export supporting CSV tables

costs_system = df_central.groupby(["Year", "Scenario"], as_index=False).agg({
    "Fuel_Exp_EUR": "sum",
    "ETS_Cost_EUR": "sum",
    "Total_Cost_EUR": "sum",
    "CO2_Emissions": "mean",
    "Avoided_CO2": "mean",
})
costs_system.to_csv("EU27_Objective2_Costs.csv", index=False)
mac_central[["Year", "MAC_EUR_per_tCO2"]].to_csv("EU27_Objective2_MAC.csv", index=False)
costs_by_seg = df_central.groupby(["Year", "Scenario", "Segment"], as_index=False).agg({
    "Fuel_Exp_EUR": "sum",
    "ETS_Cost_EUR": "sum",
    "Total_Cost_EUR": "sum",
    "CO2_Emissions": "mean",
    "Avoided_CO2": "mean",
    "Passengers": "sum",
    "Freight_tonnes": "sum",
})
costs_by_seg.to_csv("EU27_Objective2_Costs_BySegment.csv", index=False)
assumptions_df.to_csv("EU27_Objective2_Assumptions.csv", index=False)
print("Saved: EU27_Objective2_Costs.csv, EU27_Objective2_MAC.csv, EU27_Objective2_Costs_BySegment.csv, EU27_Objective2_Assumptions.csv")


Saved: EU27_Objective2_Costs.csv, EU27_Objective2_MAC.csv, EU27_Objective2_Costs_BySegment.csv, EU27_Objective2_Assumptions.csv
